In [1]:
import pandas as pd
import statsmodels.formula.api as sm
import seaborn as sns

import numpy as np
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
# preprocessing helpers
from utils import filter_words, filter_words_kids, ppp_per_usd
import unicodedata
import reverse_geocoder as rg
from statsmodels.formula.api import mixedlm
from stargazer.stargazer import Stargazer

413 filter words loaded


In [3]:
# Helper function to normalize a string
def normalize(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = unicodedata.normalize('NFKD', text)  # remove accents
    text = ''.join(c for c in text if not unicodedata.combining(c))  # remove diacritics
    return text

filter_words = list(set([normalize(word) for word in filter_words]))
filter_words_kids = list(set([normalize(word) for word in filter_words_kids]))

def add_geodata(df):
    coords = df[['lat', 'lon']].dropna()
    coord_list = list(coords.itertuples(index=False, name=None))  # [(lat, lon), ...]
    
    # Perform batch reverse geocoding
    results = rg.search(coord_list)  # returns list of dicts
    
    # Create DataFrame from results and align index with original coords
    geo_df = pd.DataFrame(results, index=coords.index)[['name', 'admin1', "cc"]]
    geo_df.columns = ['city', 'state', "country"]

    vatican_mask = (geo_df['city'] == 'Vatican City') & (geo_df['country'] == 'VA')

    # Apply corrections for Vatican City to Rome
    geo_df.loc[vatican_mask, 'city'] = 'Rome'
    geo_df.loc[vatican_mask, 'state'] = 'Latium' # Assuming 'Latium' is the desired state for Rome in this context
    geo_df.loc[vatican_mask, 'country'] = 'IT'
    
    # Merge results back into original DataFrame
    df[['city', 'state', "country"]] = geo_df


# Use the PPP for Germany to convert USD back to EUR for final reporting
USD_TO_EUR_RATE = ppp_per_usd['DE'] # 0.733194 EUR per USD

def unify_currencies(df):
    """Converts prices to a common base (USD) using PPP data and then to EUR for reporting."""
    df['price_usd'] = df.apply(
        lambda row: row['simpleCutSalePrice'] / ppp_per_usd.get(row['country'], 1.0),
        axis=1
    )
    df['price'] = df['price_usd'] * USD_TO_EUR_RATE

    # Drop the intermediate 'price_usd' column
    df = df.drop(columns=['price_usd'])

def add_average_duration(df):
    df['duration'] = (df["simpleCutDurationMin"] + df["simpleCutDurationMax"]) / 2

def preprocess(df):
    add_geodata(df)
    print(df.columns)
    unify_currencies(df)
    add_average_duration(df)
    # Filter cuts that are shorter than 10 minutes (false positives)
    df = df[df['duration'] > 10]
    df = df[df["duration"] < 120]
    df = df[df["country"] != "LU"]
    df = df[df["country"] != "SM"]
    df = df[df["price"] < 200]
    df = df[df["name"] != "Test Salon TEST PURPOSE'S ONLY"]
    return df.reset_index(drop=True)

In [4]:
df_adults = pd.read_csv('../data/treatwell_without_raw-all-2025-06-02.csv')\ndf_adults = preprocess(df_adults)
# Filter cuts that include one of the filter words
print(f"before filtering: {len(df_adults)}")
df_adults = df_adults[
    df_adults['simpleCutName'].apply(lambda x: not any(word in normalize(str(x)) for word in filter_words))
]
print(f"after filtering: {len(df_adults)}")
df_adults = df_adults[(df_adults["is_male"] == True) | (df_adults["is_female"] == True)]


df_children = pd.read_csv('../data/treatwell_without_raw_kids-2025-06-02.csv')\ndf_children = preprocess(df_children)
print(f"before filtering: {len(df_children)}")
# Filter cuts that include one of the filter words
df_children = df_children[
    df_children['simpleCutName'].apply(lambda x: not any(word in normalize(str(x)) for word in filter_words_kids))
]
print(f"after filtering: {len(df_children)}")
df_children = df_children[(df_children["is_boys"] == True) | (df_children["is_girls"] == True) | (df_children["is_unisex"] == True)]
print(len(df_children))

/var/folders/3t/wflb1ys911b01r8mn5tknndh0000gn/T/ipykernel_46586/531667268.py:1: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_adults = pd.read_csv('treatwell_without_raw-all-2025-06-02.csv')


Loading formatted geocoded file...
Index(['id', 'name', 'averageRating', 'ratingCount', 'simpleCutSalePrice',
       'simpleCutFullPrice', 'simpleCutDurationMin', 'simpleCutDurationMax',
       'simpleCutName', 'postalCode', 'adress', 'lat', 'lon', 'is_male',
       'is_female', 'type', 'city', 'state', 'country'],
      dtype='object')
before filtering: 38844
after filtering: 33683


FileNotFoundError: [Errno 2] No such file or directory: 'treatwell_without_raw_kids-2025-06-02.csv'

In [ ]:
basic_model = sm.ols(formula="price ~ is_female + duration", data=df_adults).fit(cov_type='cluster', cov_kwds={'groups': df_adults['id']})

fixed_effects = sm.ols(
    formula="price ~ ratingCount + averageRating + duration * is_female + C(state)",
    data=df_adults
).fit(cov_type='cluster', cov_kwds={'groups': df_adults['id']})


mixed_effects = mixedlm("price ~ ratingCount + averageRating + duration * is_female + C(state)", data=df_adults, groups=df_adults["id"]).fit()

mixed_effects_no_interaction = mixedlm("price ~ ratingCount + averageRating + duration + is_female + C(state)", data=df_adults, groups=df_adults["id"]).fit()

df_adults['country'] = pd.Categorical(df_adults['country'], categories=df_adults['country'].unique(), ordered=False)
df_adults['country'] = df_adults['country'].cat.set_categories(['DE'] + [cat for cat in df_adults['country'].cat.categories if cat != 'DE'], ordered=False)
mixed_effects_country_interaction = mixedlm("price ~ ratingCount + averageRating + duration + is_female * country", data=df_adults, groups=df_adults["id"]).fit()

order = [
    'Intercept',
    'is_female[T.True]',
    'averageRating',
    'ratingCount',
    'duration',
    'duration:is_female[T.True]',
    'is_female[T.True]:country[T.AT]',
    'is_female[T.True]:country[T.BE]',
    'is_female[T.True]:country[T.CH]',
    'is_female[T.True]:country[T.ES]',
    'is_female[T.True]:country[T.FR]',
    'is_female[T.True]:country[T.GB]',
    'is_female[T.True]:country[T.IT]',
    'is_female[T.True]:country[T.NL]',
    'is_female[T.True]:country[T.PT]',
    "Group Var"
]
stargazer = Stargazer([basic_model, fixed_effects, mixed_effects, mixed_effects_no_interaction, mixed_effects_country_interaction])
stargazer.custom_columns([
    "Baseline OLS",
    "FE",
    "ME (w/ Time Interaction)",
    "ME (No Interaction)",
    "ME (w/ Country Interaction)",
])
stargazer.covariate_order(order)
stargazer.rename_covariates({
    'is_female[T.True]': 'Female',
    'averageRating': 'AvgRating',
    'ratingCount': 'RatingCount',
    'duration': 'Duration',
    'duration:is_female[T.True]': 'Duration × Female',
    'Intercept': 'Const.',
    "Group Var": "Group Var", # This is usually for the random effects variance
    'is_female[T.True]:country[T.AT]': 'Female × Country (AT)',
    'is_female[T.True]:country[T.BE]': 'Female × Country (BE)',
    'is_female[T.True]:country[T.CH]': 'Female × Country (CH)',
    'is_female[T.True]:country[T.ES]': 'Female × Country (ES)',
    'is_female[T.True]:country[T.FR]': 'Female × Country (FR)',
    'is_female[T.True]:country[T.GB]': 'Female × Country (GB)',
    'is_female[T.True]:country[T.IT]': 'Female × Country (IT)',
    'is_female[T.True]:country[T.NL]': 'Female × Country (NL)',
    'is_female[T.True]:country[T.PT]': 'Female × Country (PT)'
})
stargazer.add_line('State Fixed Effects', ['No', 'Yes', 'Yes', 'Yes', 'Yes'])
stargazer.add_line('Salon Random Effects', ['No', 'No', 'Yes', 'Yes', 'Yes'])
latex_table = stargazer.render_latex()
print(latex_table)
from IPython.core.display import display, HTML
display(HTML(stargazer.render_html()))

In [ ]:
basic_model = sm.ols(formula="price ~ is_girls + is_unisex + duration", data=df_children).fit(cov_type='cluster', cov_kwds={'groups': df_children['id']})

fixed_effects = sm.ols(
    formula="price ~ ratingCount + averageRating + (duration * is_girls) + (duration * is_unisex) + C(state)",
    data=df_children
).fit(cov_type='cluster', cov_kwds={'groups': df_children['id']})

mixed_effects_children = mixedlm("price ~ ratingCount + averageRating + (duration * is_girls) + (duration * is_unisex) + C(state)", data=df_children, groups=df_children["id"]).fit()

mixed_effects_no_interaction_children = mixedlm("price ~ ratingCount + averageRating + duration + is_girls + is_unisex + C(state)", data=df_children, groups=df_children["id"]).fit()


order = [
    'Intercept',
    'is_girls[T.True]',
    'is_unisex[T.True]',
    'averageRating',
    'ratingCount',
    'duration',
    'duration:is_girls[T.True]',
    'duration:is_unisex[T.True]',
    "Group Var"
]
stargazer = Stargazer([basic_model, fixed_effects, mixed_effects_children, mixed_effects_no_interaction_children])
stargazer.custom_columns([
    "Baseline OLS",
    "FE",
    "ME (w/ Interaction)",
    "ME (No Interaction)"
])
stargazer.covariate_order(order)
stargazer.rename_covariates({
    'is_girls[T.True]': 'Female',
    'is_unisex[T.True]': 'Unisex',
    'averageRating': 'AvgRating',
    'ratingCount': 'RatingCount',
    'duration': 'Duration',
    'duration:is_girls[T.True]': 'Duration × Female',
    'duration:is_unisex[T.True]': 'Duration × Unisex',
    'Intercept': 'Const.',
    "Group Var": "Group Var"
})
stargazer.add_line('State Fixed Effects', ['No', 'Yes', 'Yes', 'Yes'])
stargazer.add_line('Salon Random Effects', ['No', 'No', 'Yes', 'Yes'])
latex_table = stargazer.render_latex()
print(latex_table)
display(HTML(stargazer.render_html()))

In [ ]:
plt.figure(figsize=(6, 5))
sns.boxplot(
    data=df_adults,
    x='is_female',
    y='price'
)
plt.xticks([0, 1], ['Men', 'Women'])
plt.ylabel("Price (euro)")
plt.xlabel("Gender")
plt.title("Distribution of Haircut Prices by Gender")
plt.tight_layout()
plt.show()

(df_adults[(df_adults["is_male"] == False) & (df_adults["is_female"] == False)])
df_adults[df_adults["price"] > 150]

In [ ]:
df_children_plot = df_children.copy()  # Avoid modifying original
df_children_plot["gender"] = df_children_plot.apply(
    lambda row: "Boys" if row["is_boys"] else ("Girls" if row["is_girls"] else "Unisex"),
    axis=1
)
plt.figure(figsize=(6, 5))
sns.boxplot(
    data=df_children_plot,
    x='gender',
    y='price',
    order=["Boys", "Girls", "Unisex"]  # consistent order
)
plt.ylabel("Price (euro)")
plt.xlabel("Gender")
plt.title("Distribution of Children's Haircut Prices by Gender")
plt.tight_layout()
plt.show()

In [ ]:
# Simulated duration range (in minutes)
duration = np.linspace(10, 60, 100)

# Coefficients from Model (4): Mixed Effects with Interaction
intercept = mixed_effects_children.params['Intercept']
female_coef = mixed_effects_children.params['is_girls[T.True]']
duration_coef = mixed_effects_children.params['duration']
interaction_coef = mixed_effects_children.params['duration:is_girls[T.True]']

# Predicted prices
price_boys = intercept + duration_coef * duration
price_girls = intercept + female_coef + (duration_coef + interaction_coef) * duration

# Plot
plt.figure(figsize=(8, 5))
plt.plot(duration, price_boys, label="Boys' Haircuts", color="blue")
plt.plot(duration, price_girls, label="Girls' Haircuts", color="orange")
plt.fill_between(duration, price_boys, price_girls, where=(price_girls > price_boys), 
                 color='orange', alpha=0.2, label="Price Gap")

plt.xlabel("Duration (minutes)")
plt.ylabel("Predicted Price (€)")
plt.title("Interaction Effect of Duration and Gender on Child Haircut Prices")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Simulated duration range (in minutes)
duration = np.linspace(10, 60, 100)

# Coefficients from Model (4): Mixed Effects with Interaction
intercept = mixed_effects.params['Intercept']
female_coef = mixed_effects.params['is_female[T.True]']
duration_coef = mixed_effects.params['duration']
interaction_coef = mixed_effects.params['duration:is_female[T.True]']

# Predicted prices
price_men = intercept + duration_coef * duration
price_women = intercept + female_coef + (duration_coef + interaction_coef) * duration

# Plot
plt.figure(figsize=(8, 5))
plt.plot(duration, price_men, label="Men Haircuts", color="blue")
plt.plot(duration, price_women, label="Women Haircuts", color="orange")
plt.fill_between(duration, price_men, price_women, where=(price_women > price_men), 
                 color='orange', alpha=0.2, label="Price Gap")

plt.xlabel("Duration (minutes)")
plt.ylabel("Predicted Price (€)")
plt.title("Interaction Effect of Duration and Gender on Adult Haircut Prices")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# calculate price means
print(df_adults["price"].mean())
print(df_children["price"].mean())

print("percentages")
print(mixed_effects_no_interaction.params['is_female[T.True]'])
print((mixed_effects_no_interaction.params['is_female[T.True]'] / df_adults["price"].mean()) * 100)
print(mixed_effects_no_interaction_children.params['is_girls[T.True]'])
print((mixed_effects_no_interaction_children.params['is_girls[T.True]'] / df_children["price"].mean())*100)

In [ ]:
import pandas as pd
from scipy import stats # Import the stats module for t-tests

# Assuming df_adults and df_children DataFrames are already created and contain the necessary columns:
# 'price', 'duration', 'averageRating', 'ratingCount', 'is_female', 'is_girls', 'is_boys', 'is_unisex'

stats_cols = ['price', 'duration', 'averageRating', 'ratingCount']

def summarize_and_ttest_adults(df_adults):
    """
    Computes descriptive statistics and performs independent t-tests
    for the difference between Women and Men adult groups.
    """
    # Separate data for Women and Men
    df_women = df_adults[df_adults['is_female']].copy() # Use .copy() to avoid SettingWithCopyWarning
    df_men = df_adults[~df_adults['is_female']].copy() # Use .copy()

    summary_data = {}

    # Compute descriptive stats for Women and Men
    for col in stats_cols:
        summary_data[f'{col}_Women_mean'] = df_women[col].mean()
        summary_data[f'{col}_Women_std'] = df_women[col].std()
        summary_data[f'{col}_Men_mean'] = df_men[col].mean()
        summary_data[f'{col}_Men_std'] = df_men[col].std()

        # Perform independent samples t-test
        # We use equal_var=False (Welch's t-test) which does not assume equal variances
        # This is generally safer unless you have strong evidence of equal variances.
        # dropna=True is used to handle potential NaN values in the columns
        t_stat, p_value = stats.ttest_ind(
            df_women[col].dropna(),
            df_men[col].dropna(),
            equal_var=False
        )
        summary_data[f'{col}_diff_mean'] = df_women[col].mean() - df_men[col].mean()
        summary_data[f'{col}_t_stat'] = t_stat
        summary_data[f'{col}_p_value'] = p_value

    # Convert to DataFrame for better presentation
    adult_stats_df = pd.DataFrame([summary_data])

    # Reshape DataFrame for a more readable format (optional but recommended)
    # This part restructures the DataFrame to have rows for each variable and columns for stats
    output_rows = []
    for col in stats_cols:
        row_data = {
            'Variable': col,
            'Women_mean': summary_data[f'{col}_Women_mean'],
            'Women_std': summary_data[f'{col}_Women_std'],
            'Men_mean': summary_data[f'{col}_Men_mean'],
            'Men_std': summary_data[f'{col}_Men_std'],
            'Diff_mean (W - M)': summary_data[f'{col}_diff_mean'],
            'T-statistic': summary_data[f'{col}_t_stat'],
            'P-value': summary_data[f'{col}_p_value']
        }
        output_rows.append(row_data)

    adult_stats_formatted = pd.DataFrame(output_rows)

    # Add counts
    adult_stats_formatted['Women_count'] = len(df_women)
    adult_stats_formatted['Men_count'] = len(df_men)


    return adult_stats_formatted.round(3) # Round to 3 decimal places for presentation


def summarize_and_ttest_children(df_children):
    """
    Computes descriptive statistics and performs independent t-tests
    for the difference between Girls and Boys children groups.
    Also includes descriptive stats for Unisex group.
    """
    # Separate data for Girls, Boys, and Unisex
    df_girls = df_children[df_children['is_girls']].copy() # Use .copy()
    df_boys = df_children[df_children['is_boys']].copy()   # Use .copy()
    df_unisex = df_children[df_children['is_unisex']].copy() # Use .copy()


    summary_data = {}

    # Compute descriptive stats for Girls, Boys, and Unisex
    for col in stats_cols:
        summary_data[f'{col}_Girls_mean'] = df_girls[col].mean()
        summary_data[f'{col}_Girls_std'] = df_girls[col].std()
        summary_data[f'{col}_Boys_mean'] = df_boys[col].mean()
        summary_data[f'{col}_Boys_std'] = df_boys[col].std()
        summary_data[f'{col}_Unisex_mean'] = df_unisex[col].mean()
        summary_data[f'{col}_Unisex_std'] = df_unisex[col].std()

        # Perform independent samples t-test for Girls vs. Boys
        # dropna=True is used to handle potential NaN values in the columns
        t_stat, p_value = stats.ttest_ind(
            df_girls[col].dropna(),
            df_boys[col].dropna(),
            equal_var=False
        )
        summary_data[f'{col}_diff_mean (G - B)'] = df_girls[col].mean() - df_boys[col].mean()
        summary_data[f'{col}_t_stat (G vs B)'] = t_stat
        summary_data[f'{col}_p_value (G vs B)'] = p_value


    # Convert to DataFrame for better presentation
    # This part restructures the DataFrame to have rows for each variable and columns for stats
    output_rows = []
    for col in stats_cols:
        row_data = {
            'Variable': col,
            'Girls_mean': summary_data[f'{col}_Girls_mean'],
            'Girls_std': summary_data[f'{col}_Girls_std'],
            'Boys_mean': summary_data[f'{col}_Boys_mean'],
            'Boys_std': summary_data[f'{col}_Boys_std'],
            'Unisex_mean': summary_data[f'{col}_Unisex_mean'],
            'Unisex_std': summary_data[f'{col}_Unisex_std'],
            'Diff_mean (G - B)': summary_data[f'{col}_diff_mean (G - B)'],
            'T-statistic (G vs B)': summary_data[f'{col}_t_stat (G vs B)'],
            'P-value (G vs B)': summary_data[f'{col}_p_value (G vs B)']
        }
        output_rows.append(row_data)

    child_stats_formatted = pd.DataFrame(output_rows)

    # Add counts
    child_stats_formatted['Girls_count'] = len(df_girls)
    child_stats_formatted['Boys_count'] = len(df_boys)
    child_stats_formatted['Unisex_count'] = len(df_unisex)


    return child_stats_formatted.round(3) # Round to 3 decimal places for presentation



adult_stats_with_ttest = summarize_and_ttest_adults(df_adults.copy()) # Use .copy()
children_stats_with_ttest = summarize_and_ttest_children(df_children.copy()) # Use .copy()

print("Adult Haircut Stats with T-tests:\n", adult_stats_with_ttest)
print("\nChildren Haircut Stats with T-tests (Girls vs Boys):\n", children_stats_with_ttest)


In [ ]:
mixed_effects_country_interaction = mixedlm("price ~ ratingCount + averageRating + duration + is_female * country", data=df_adults, groups=df_adults["id"]).fit()
mixed_effects_country_interaction.summary()

In [ ]:
print(len(df_children[(df_children["is_boys"] == True) | (df_children["is_girls"] == True) | (df_children["is_unisex"] == True)]))
df_children[(df_children["is_boys"] == False) & (df_children["is_girls"] == False) & (df_children["is_unisex"] == False)]

In [16]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.oaxaca import OaxacaBlinder

# 1. Create Dummy Variables for Country
# drop_first=True avoids the dummy variable trap
country_dummies = pd.get_dummies(df_adults['country'], prefix='country', drop_first=True, dtype=int)

# 2. Define your continuous/ordinal controls
# Do NOT include salon_id here, but we'll add is_female separately for bifurcation
continuous_features = ['ratingCount', 'averageRating', 'duration']

# 3. Combine everything into your Exogenous Matrix (X)
X = df_adults[continuous_features].join(country_dummies)

# 4. Add the constant (Intercept)
X = sm.add_constant(X)

# 5. Add is_female as a column for bifurcation (but don't include it in the model)
# We'll remove it after the decomposition
X['is_female'] = df_adults['is_female'].astype(int)

# 6. Run the Decomposition
# Pass bifurcate as column name when exog is a DataFrame
ob = OaxacaBlinder(endog=df_adults['price'], 
                   exog=X, 
                   bifurcate='is_female')

results = ob.two_fold()

# 6. Print Summary
print(results.summary())

Oaxaca-Blinder Two-fold Effects
Unexplained Effect: 3.96737
Explained Effect: 5.24541
Gap: 9.21277
None


In [17]:
# Oaxaca-Blinder Decomposition: Only Salons with Both Genders
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.oaxaca import OaxacaBlinder

# Filter to only salons that have both genders
# Find salons that have both male and female haircuts
salons_with_both_genders = df_adults.groupby('id')['is_female'].agg(['nunique', 'sum', 'count']).reset_index()
salons_with_both_genders = salons_with_both_genders[
    (salons_with_both_genders['nunique'] == 2) &  # Has both True and False
    (salons_with_both_genders['sum'] > 0) &  # Has at least one female
    (salons_with_both_genders['sum'] < salons_with_both_genders['count'])  # Has at least one male
]['id'].tolist()

# Filter the dataframe to only these salons
df_adults_both_genders = df_adults[df_adults['id'].isin(salons_with_both_genders)].copy()
print(f"Total salons: {df_adults['id'].nunique()}")
print(f"Salons with both genders: {len(salons_with_both_genders)}")
print(f"Total observations: {len(df_adults)}")
print(f"Observations in salons with both genders: {len(df_adults_both_genders)}")
print()

# 1. Create Dummy Variables for Country
# drop_first=True avoids the dummy variable trap
country_dummies = pd.get_dummies(df_adults_both_genders['country'], prefix='country', drop_first=True, dtype=int)

# 2. Define your continuous/ordinal controls
# Do NOT include salon_id here, but we'll add is_female separately for bifurcation
continuous_features = ['ratingCount', 'averageRating', 'duration']

# 3. Combine everything into your Exogenous Matrix (X)
X = df_adults_both_genders[continuous_features].join(country_dummies)

# 4. Add the constant (Intercept)
X = sm.add_constant(X)

# 5. Add is_female as a column for bifurcation (but don't include it in the model)
X['is_female'] = df_adults_both_genders['is_female'].astype(int)

# 6. Run the Decomposition
# Pass bifurcate as column name when exog is a DataFrame
ob_both_genders = OaxacaBlinder(endog=df_adults_both_genders['price'], 
                                exog=X, 
                                bifurcate='is_female')

results_both_genders = ob_both_genders.two_fold()

# 7. Print Summary
print("Oaxaca-Blinder Decomposition (Salons with Both Genders Only):")
print(results_both_genders.summary())


Total salons: 13659
Salons with both genders: 6459
Total observations: 33683
Observations in salons with both genders: 20758

Oaxaca-Blinder Decomposition (Salons with Both Genders Only):
Oaxaca-Blinder Two-fold Effects
Unexplained Effect: 4.59609
Explained Effect: 7.57522
Gap: 12.17131
None


In [18]:
import matplotlib.pyplot as plt
import pandas as pd

# Assuming 'results' is your OaxacaBlinder result object from the previous step
# Extract the scalar values
unexplained = results.unexplained
explained = results.explained
gap = results.gap

# Create a DataFrame for plotting
plot_data = pd.DataFrame({
    'Component': ['Explained (Duration/Country)', 'Unexplained (Pink Tax)'],
    'Amount': [explained, unexplained]
})

# Calculate percentage for labels
total_gap = explained + unexplained
plot_data['Percentage'] = (plot_data['Amount'] / total_gap) * 100

# Plot
fig, ax = plt.subplots(figsize=(6, 6))
bars = ax.bar(plot_data['Component'], plot_data['Amount'], color=['#bdc3c7', '#e74c3c'])

# Add labels on top of bars
for bar, pct, amt in zip(bars, plot_data['Percentage'], plot_data['Amount']):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'€{amt:.2f}\n({pct:.1f}%)',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

# Styling
ax.set_ylabel('Price Gap (€)')
ax.set_title('Decomposition of the Gender Price Gap', fontsize=14)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

AttributeError: 'OaxacaResults' object has no attribute 'unexplained'